In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import uproot
import sys
import math

dir = "/Users/alexanderantonakis/Desktop/Software/AFrameAnalysis/Macros/"

sys.path.append("../Utils")
sys.path.append("../Configs")

from ChannelMap import ChannelMap

frame = 1
strip_w = 11.2 # cm 

data_file = "../Data/outtree_frame1_run6106_run6110.root"
cluster_file = "clusters_frame1.root"
config = "config_frame1.txt"
run_config = "run_config_frame1.txt"

fig_dir = "../Figs/Frame"+str(frame)+"/"

# set up the geometry of the frame
map = ChannelMap("../Configs/"+config, "../Configs/"+run_config)
map.initialize_config()
map.initialize_run_config()
map.calculate_params()
print("initialized the geometry and voltages")
print("")

# initialize the horizontal febs --> useful to have
febs = map.mac5
horiz_febs = []
for feb in febs:
    if map.is_horiz(feb):
        horiz_febs.append(feb)
        
print("All Horizontal FEBs in this file:", horiz_febs)




In [ ]:
import ROOT
# Open the ROOT file and the TTree
file = uproot.open(data_file)  # Replace with your ROOT file
tree = file["reco/events"]    

run_data = tree["fRun"].array(library="np")


def get_feb_hits(row):
    f = row["flags"]
    m = row["mac5"]
    d= np.array(f) % 3 + np.array(m)
    d = list(d)
    counts = []
    for feb in horiz_febs:
        counts.append(d.count(feb))
    return counts

branches = ["fRun", "flags", "mac5"]  # Add the branches you want

#h_list = [ROOT.TH1D("h_feb"+str(feb), "", 1, 0, 1) for feb in horiz_febs]

N_vals = [0 for num in range(len(horiz_febs))]

N_vals_all = []

corrections = [0 for num in range(len(map.runs))]
count = 0
for run in map.runs:
    mask = (run_data == run)
    df = tree.arrays(branches, entry_start=mask.nonzero()[0][0], entry_stop=mask.nonzero()[0][-1] + 1, library="pd")
    df["feb_counts"] = df.apply(get_feb_hits, axis=1)

    # Make DataFrames that are easier to work with
    hit_df = pd.DataFrame(df['feb_counts'].tolist())
    columns = []
    for feb in horiz_febs:
        columns.append("FEB"+str(feb))
    hit_df.columns = columns
    
    for num in range(len(horiz_febs)):
        #h_list[num].Fill(sumlist(hit_df["FEB"+str(horiz_febs[num])].values)))
        N_vals[num] = sum(list(hit_df["FEB"+str(horiz_febs[num])].values))
        
    N_vals = np.array(N_vals)
    N_vals_all.append(N_vals)
    corrections[count] = N_vals / N_vals[horiz_febs.index(156)]
    
    N_vals = np.zeros_like(N_vals)
    count += 1


print(corrections)

In [ ]:
for corr in corrections:
    print(corr)

In [ ]:
for num in N_vals_all:
    print(num)

In [ ]:
# Open the ROOT file and the TTree
file = uproot.open(cluster_file)  # Replace with your ROOT file
tree = file["cluster_tree"]    

# Convert the TTree to a pandas DataFrame
df = tree.arrays(library="pd")

# Make DataFrames that are easier to work with
strip_df = pd.DataFrame(df['strips'].tolist())
columns = ["Run"]
for feb in horiz_febs:
    columns.append("FEB"+str(feb))
strip_df.columns = columns


strip_df[:4]

In [ ]:
Nc_vals = [0 for num in range(len(horiz_febs))]
Nc_vals_all = []

for run in map.runs:
    new_df = strip_df.query("Run == "+str(run))
    count = 0
    for feb in horiz_febs:
        s = new_df["FEB"+str(feb)].values
        Nc = 0
        for num in s:
            if num != -1:
                Nc += 1
        Nc_vals[count] = Nc
        
        count += 1
    Nc_vals_all.append(Nc_vals)
    Nc_vals = [0 for num in range(len(horiz_febs))]

for num in Nc_vals_all:
    print(num)

In [ ]:

corr = []
for arr in Nc_vals_all:
    c = np.ones_like(np.array(arr))*float(arr[horiz_febs.index(182)])
    #print(c)
    c /= np.array(arr)
    corr.append(c)


for arr in corr:
    print(arr)
    

In [ ]:
# save the correction to a ROOT TNtuple

feb_str = ""
count = 0
for feb in horiz_febs:
    feb_str += "FEB"
    feb_str += str(feb)
    if count == len(horiz_febs) -1:
        break
    count += 1
    feb_str+= ":"
    
corr_tree = ROOT.TNtuple("corr_tree", "corr_tree", "Run:"+feb_str)
count = 0
for run in map.runs:
    stuff = [run]
    for num in range(len(horiz_febs)):
        stuff.append(corr[count][num])
    corr_tree.Fill(*stuff)
    count += 1

corr_tree.SetDirectory(0)

outfile = ROOT.TFile("frame1_corrections.root", "RECREATE")
outfile.cd()
corr_tree.Write()

outfile.Close()
print("finished")

In [ ]:
eff_sel = []
I = horiz_febs.index(156)
for num in range(len(map.runs)):
    eff_sel.append(Nc_vals_all[num][I] / N_vals_all[num][I])

print(eff_sel)

In [ ]:
eff_h_all = []
eff_h_curr = [0 for num in range(len(horiz_febs))]

for r in range(len(map.runs)):
    for num in range(len(horiz_febs)):
        eff_h_curr[num] = (Nc_vals_all[r][num] / (N_vals_all[r][num]*eff_sel[r]))
    eff_h_all.append(eff_h_curr)
    eff_h_curr = [0 for num in range(len(horiz_febs))]

for num in eff_h_all:
    print(num)

In [ ]:
print(1/0.3)

In [ ]:


norm_N = []
for num in range(len(N_vals_all)):
    A = N_vals_all[num][-1]
    B = np.array(N_vals_all[num])/A
    norm_N.append(B)

print("Divide N by the true N")
for arr in norm_N:
    print(arr)



In [ ]:
print("Nc / N")

ratios = []
for num in range(len(N_vals_all)):
    A = np.array(N_vals_all[num])
    B = np.array(Nc_vals_all[num])
    ratios.append(B/A)
for arr in ratios:
    print(arr)
    

In [ ]:
C = []
for arr in ratios:
    C.append(np.ones_like(arr)*max(arr)/arr)

for arr in C:
    print(arr)

In [ ]:
for arr in ratios:
    arr /= 0.33
    print(arr)
    

In [ ]:
for run in map.runs:
    print(run)